In [28]:
# =========================================================
# SQL Analytics on E-Commerce Dataset using SQLite
# This notebook performs business analysis using SQL queries
# including Joins, Aggregations, Window Functions, CTEs,
# Customer Segmentation and RFM Analysis.
# =========================================================

In [29]:
# Import required libraries
import sqlite3
import pandas as pd
# Connect to SQLite Database
conn = sqlite3.connect("../ecommerce.db")

In [30]:
# Query 1: Calculate Total Revenue

query = """
SELECT
SUM(quantity * unit_price * (1-discount_percent/100.0)) AS Total_Revenue
FROM order_items;
"""

pd.read_sql(query, conn)

,Total_Revenue
0,1.679432e+09


In [31]:
# Query 2: Revenue Generated by Each Customer

query = """
SELECT
product_id,
SUM(quantity) AS Total_Quantity
FROM order_items
GROUP BY product_id
ORDER BY Total_Quantity DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,product_id,Total_Quantity
0,P394,273
1,P368,265
2,P198,263
3,P381,251
4,P460,250
5,P203,248
6,P266,247
7,P110,246
8,P009,246
9,P428,245


In [32]:
# Query 3: Revenue by Product Category

query = """
SELECT
o.customer_id,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) AS Revenue
FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY o.customer_id
ORDER BY Revenue DESC
LIMIT 10;
"""

pd.read_sql(query, conn)

,customer_id,Revenue
0,Unknown,79524973.38
1,C610,4021638.92
2,C302,4020298.49
3,C512,3809265.79
4,C868,3705627.77
5,C418,3685404.10
6,C211,3657937.19
7,C732,3620474.11
8,C533,3554307.90
9,C049,3524604.88


In [33]:
# Query 4: Monthly Revenue Trend
query = """
SELECT
p.category,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) AS Revenue
FROM order_items oi
JOIN products p
ON oi.product_id=p.product_id
GROUP BY p.category;
"""

pd.read_sql(query, conn)

,category,Revenue
0,Books,3.583174e+08
1,Clothing,4.258517e+08
2,Electronics,3.876815e+08
3,Home,5.075819e+08


In [34]:
# Query 5: Identify Top Products by Quantity Sold and Revenue

query = """
SELECT
substr(order_date,1,7) AS Month,
SUM(quantity*unit_price*(1-discount_percent/100.0)) AS Revenue
FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id
GROUP BY Month
ORDER BY Month;
"""

pd.read_sql(query, conn)

,Month,Revenue
0,2024-07,57143320.29
1,2024-08,73795470.65
2,2024-09,76557077.99
3,2024-10,74841320.69
4,2024-11,61525216.12
5,2024-12,73204295.14
6,2025-01,69010097.53
7,2025-02,60955252.66
8,2025-03,71155697.34
9,2025-04,64022519.75


In [35]:
# Query 6: Calculate Average Order Value (AOV) for Each Customer

query = """
SELECT
o.region_code,
substr(o.order_date,1,10) AS order_date,
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0)) AS daily_revenue,

SUM(
SUM(oi.quantity*oi.unit_price*(1-oi.discount_percent/100.0))
) OVER(
PARTITION BY o.region_code
ORDER BY substr(o.order_date,1,10)
) AS running_total

FROM orders o
JOIN order_items oi
ON o.order_id=oi.order_id

GROUP BY o.region_code, substr(o.order_date,1,10);
"""

pd.read_sql(query, conn)

,region_code,order_date,daily_revenue,running_total
0,East,2024-07-08,62323.80,6.232380e+04
1,East,2024-07-09,635831.08,6.981549e+05
2,East,2024-07-10,246824.36,9.449792e+05
3,East,2024-07-11,575461.96,1.520441e+06
4,East,2024-07-12,391852.26,1.912293e+06
...,...,...,...,...
2803,West,2026-07-02,1479081.06,4.354658e+08
2804,West,2026-07-04,683613.12,4.361494e+08
2805,West,2026-07-05,554887.07,4.367043e+08
2806,West,2026-07-06,773208.36,4.374775e+08


In [36]:
# Query 7: Rank Customers by Lifetime Value using RANK()

query = """
...
"""

query = """
SELECT *

FROM(

SELECT

p.category,

p.product_name,

SUM(oi.quantity*oi.unit_price) AS total_revenue,

DENSE_RANK() OVER(

PARTITION BY p.category

ORDER BY SUM(oi.quantity*oi.unit_price) DESC

) AS rank_in_category

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY p.category,p.product_name

);

"""

pd.read_sql(query,conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Notebook,242609979,1
1,Books,Book,206942671,2
2,Books,Laptop,29588957,3
3,Clothing,T-Shirt,177209545,1
4,Clothing,Shoes,125458539,2
5,Clothing,Jacket,124690132,3
6,Clothing,Jeans,108542558,4
7,Clothing,Laptop,30225667,5
8,Electronics,Keyboard,76237842,1
9,Electronics,Mobile,71908386,2


In [37]:
# Query 8: Calculate Running Total of Monthly Revenue

query = """
SELECT
customer_id,

SUM(quantity*unit_price) AS lifetime_value,

RANK() OVER(

ORDER BY SUM(quantity*unit_price) DESC

) AS customer_rank

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY customer_id;
"""

pd.read_sql(query,conn)

,customer_id,lifetime_value,customer_rank
0,Unknown,106591266,1
1,C610,5361105,2
2,C512,5137535,3
3,C302,5109716,4
4,C868,5022051,5
...,...,...,...
996,C848,361872,997
997,C068,354869,998
998,C881,341896,999
999,C213,338292,1000


In [38]:
# Query 9: Calculate Moving Average of Monthly Revenue

query="""

SELECT

customer_id,

COUNT(order_id) AS total_orders,

CASE

WHEN COUNT(order_id)=1 THEN 'One Time'

WHEN COUNT(order_id)<=5 THEN 'Occasional'

ELSE 'Loyal'

END AS Customer_Type

FROM orders

GROUP BY customer_id;

"""

pd.read_sql(query,conn)

,customer_id,total_orders,Customer_Type
0,C001,5,Occasional
1,C002,7,Loyal
2,C003,10,Loyal
3,C004,7,Loyal
4,C005,13,Loyal
...,...,...,...
996,C996,7,Loyal
997,C997,11,Loyal
998,C998,7,Loyal
999,C999,11,Loyal


In [39]:
# Query 10: Calculate Monthly Revenue Growth using CTE

query="""

SELECT

o.customer_id,

SUM(oi.quantity*oi.unit_price) AS TotalSpend,

CASE

WHEN SUM(oi.quantity*oi.unit_price)>500000 THEN 'High'

WHEN SUM(oi.quantity*oi.unit_price)>200000 THEN 'Medium'

ELSE 'Low'

END AS SpendTier

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY o.customer_id;

"""

pd.read_sql(query,conn)

,customer_id,TotalSpend,SpendTier
0,C001,1114704,High
1,C002,1872581,High
2,C003,1897372,High
3,C004,992073,High
4,C005,4025597,High
...,...,...,...
996,C996,1417831,High
997,C997,2613574,High
998,C998,1879053,High
999,C999,2194957,High


In [40]:
# Query 11: Perform RFM Analysis (Recency, Frequency, Monetary)

query = """

SELECT

o.customer_id,

MAX(o.order_date) AS LastOrder,

COUNT(DISTINCT o.order_id) AS Frequency,

SUM(oi.quantity * oi.unit_price * (1 - oi.discount_percent/100.0)) AS Monetary

FROM orders o

JOIN order_items oi
ON o.order_id = oi.order_id

GROUP BY o.customer_id;

"""

pd.read_sql(query, conn)

,customer_id,LastOrder,Frequency,Monetary
0,C001,2026-02-12 04:30:49,5,825569.99
1,C002,2026-06-28 04:09:09,7,1490106.19
2,C003,2026-07-05 20:00:06,9,1404324.81
3,C004,2026-04-27 00:00:30,6,760721.68
4,C005,2026-06-08 15:10:53,13,3221851.73
...,...,...,...,...
996,C996,2026-02-27 13:49:15,7,1113559.53
997,C997,2026-05-20 20:30:36,11,1918834.62
998,C998,2026-06-29 14:29:07,7,1463998.17
999,C999,2026-05-22 20:32:43,10,1436052.65


In [41]:
# Query 12: Segment Customers into One-Time, Occasional and Loyal Categories

query="""

WITH MonthlyRevenue AS(

SELECT

substr(order_date,1,7) Month,

SUM(quantity*unit_price) Revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY Month

)

SELECT *

FROM MonthlyRevenue;

"""

pd.read_sql(query,conn)

,Month,Revenue
0,2024-07,76399392
1,2024-08,98351726
2,2024-09,101843186
3,2024-10,99967091
4,2024-11,83067925
5,2024-12,96723311
6,2025-01,91578895
7,2025-02,81066424
8,2025-03,94843381
9,2025-04,85925751
